# OR shorts: opening activity and range structure

This uses the later 30-minute OR short replay: 1-minute close breakout signal, OR-line limit waiting up to 15 minutes or next-candle-open entry, ticker opening gap below -1%, QQQ opening gap within ±1%, with the primary view restricted to fills before noon while QQQ is below its open. Stop/target plans remain independent. The main displayed plan is 1% stop / 2% target; 5 bps slippage per side is already in the source ledger. We analyze the same fills, without resimulating entries.

**Features known at 10:00 ET:** 30-minute opening share and dollar volume, relative share and dollar volume compared with the *same 30-minute window* in up to ten previous complete sessions (at least five required), closing location inside the OR, and directional body divided by OR width. No current or later session is included in reference volume. VWAP at 10:00 is recorded for future OR↔VWAP path studies, but is not used to select a group here. Premarket high/low and bar coverage are audited; they are not treated as predictive unless adequate premarket data exists. These comparisons are exploratory and require new-week validation.

Inputs: Backtesting 1-minute CSVs and the existing Opening_Range_Stop_Target_Grid/short_grid_with_qqq_entry.csv in My Drive. Outputs: Opening_Range_Activity_Structure_Study.

In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display
import pandas as pd,numpy as np,re
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive')
ROOT=next((p for p in (BASE/'backtesting',BASE/'Backtesting') if p.is_dir()),None)
SRC=BASE/'Opening_Range_Stop_Target_Grid'/'short_grid_with_qqq_entry.csv'
if ROOT is None or not SRC.exists():raise FileNotFoundError('Check Backtesting folder and source short grid')
OUT=BASE/'Opening_Range_Activity_Structure_Study';OUT.mkdir(exist_ok=True)
EXCLUDED={'2026-11-04','2026-11-05'}
a=pd.read_csv(SRC,dtype={'ticker':str,'date':str},low_memory=False)
assert {'ticker','date','side','boundary','entry_method','stop_pct','target_pct','before_noon','qqq_move_at_entry_pct','R','result'}.issubset(a.columns)
assert a.side.eq('short').all()
a['primary']=a.before_noon.astype(str).str.lower().eq('true') & a.qqq_move_at_entry_pct.lt(0)
print('Source fills:',len(a),'primary fills across separate plans:',a.primary.sum())

## Build one ticker-day feature row

Files are grouped by their internal timestamps because CSV filenames can be pull dates. Complete 9:30–9:59 windows require 30 distinct minute bars with valid OHLCV. Volume reference uses the preceding ten such days per ticker; a feature becomes available with at least five. Relative dollar volume uses the sum of minute volume × typical price, not today's future volume. Premarket coverage is reported in two windows (04:00–09:29 and 08:00–09:29); sparse bars can reflect missing extended-hours data rather than no trading.

In [ ]:
files={}
needed=set(a.ticker.unique())
for p in ROOT.rglob('*_1m_*.csv'):
    m=re.match(r'(.+)_1m_\\d{4}-\\d{2}-\\d{2}',p.stem)
    if m and m.group(1).upper() in needed:files.setdefault(m.group(1).upper(),[]).append(p)
print('Tickers with minute files:',len(files),'of',len(needed))
rows=[]
for n,ticker in enumerate(sorted(files),1):
    parts=[]
    for p in files[ticker]:
        try:
            raw=pd.read_csv(p)
            tc=next((c for c in raw if c.lower() in ('datetime','date','timestamp')),raw.columns[0])
            t=pd.to_datetime(raw[tc],utc=True,errors='coerce').dt.tz_convert('America/New_York')
            z=pd.DataFrame({'t':t})
            for col in ('Open','High','Low','Close','Volume'):z[col]=pd.to_numeric(raw[col],errors='coerce')
            parts.append(z.dropna(subset=['t','Open','High','Low','Close']))
        except Exception as e:print('Skipped',p.name,str(e)[:100])
    if not parts:continue
    d=pd.concat(parts).drop_duplicates('t',keep='last').sort_values('t')
    d['date']=d.t.dt.strftime('%Y-%m-%d')
    d['clock']=d.t.dt.strftime('%H:%M')
    hist=[]
    for date,g in d.groupby('date',sort=True):
        o=g[(g.clock>='09:30')&(g.clock<='09:59')]
        pm=g[(g.clock>='04:00')&(g.clock<'09:30')]
        pm_late=pm[pm.clock>='08:00']
        if o.clock.nunique()!=30 or o[['Volume']].isna().any().any() or (o.Volume<0).any():continue
        vol=float(o.Volume.sum())
        dollar=float((o.Volume*(o.High+o.Low+o.Close)/3).sum())
        hist10=hist[-10:]
        hi=float(o.High.max());lo=float(o.Low.min());width=hi-lo
        opening=float(o.iloc[0].Open);closing=float(o.iloc[-1].Close)
        cum=o.Volume.sum()
        row=dict(ticker=ticker,date=date,or_high=hi,or_low=lo,or_open=opening,or_close=closing,
                 or_width=width,or_volume=vol,or_dollar_volume=dollar,
                 prior_complete_mornings=len(hist10),
                 rvol30=vol/np.mean([v for v,db in hist10]) if len(hist10)>=5 and np.mean([v for v,db in hist10])>0 else np.nan,
                 relative_dollar_volume=dollar/np.mean([db for v,db in hist10]) if len(hist10)>=5 and np.mean([db for v,db in hist10])>0 else np.nan,
                 close_location=(closing-lo)/width if width>0 else np.nan,
                 directionality=(closing-opening)/width if width>0 else np.nan,
                 vwap_at_or_close=float((o.Volume*(o.High+o.Low+o.Close)/3).sum()/cum) if cum>0 else np.nan,
                 premarket_bars=len(pm),premarket_late_bars=len(pm_late),
                 premarket_high=float(pm.High.max()) if len(pm) else np.nan,
                 premarket_low=float(pm.Low.min()) if len(pm) else np.nan)
        rows.append(row)
        hist.append((vol,dollar))
    if n%100==0:print('Processed',n,'/',len(files))
features=pd.DataFrame(rows)
if features.empty:raise RuntimeError('No complete OR windows found')
assert not features.duplicated(['ticker','date']).any()
features.to_csv(OUT/'ticker_day_features.csv',index=False)
print('Complete OR ticker-days:',len(features),'with RVOL:',features.rvol30.notna().sum())
print('Premarket coverage among complete OR days:')
display(features.assign(pm_any=features.premarket_bars.gt(0),pm_30=features.premarket_bars.ge(30),
                        pm_late_30=features.premarket_late_bars.ge(30)).agg(
    {'pm_any':'sum','pm_30':'sum','pm_late_30':'sum'}))

## Apply features to the same short trade plans

Small predefined bands: RVOL <0.75, 0.75–<1.5, ≥1.5; OR close location bottom third, middle third, top third; directionality ≤-0.5, -0.5–0, 0–0.5, ≥0.5. For shorts, bottom-third close and negative directionality indicate an opening that finished toward the low. Separate missing reference history. Compare each entry plan with its own baseline and report dates/ticker-days. A filled-trade summary does not establish signal frequency.

In [ ]:
b=a.merge(features,on=['ticker','date'],how='left',validate='many_to_one')
b['feature_ready']=b.rvol30.notna() & b.close_location.notna()
b['rvol_band']=pd.cut(b.rvol30,[0,.75,1.5,np.inf],labels=['<0.75','0.75–<1.5','≥1.5'],right=False)
b['dollar_rvol_band']=pd.cut(b.relative_dollar_volume,[0,.75,1.5,np.inf],labels=['<0.75','0.75–<1.5','≥1.5'],right=False)
b['close_band']=pd.cut(b.close_location,[-.000001,1/3,2/3,1.000001],labels=['bottom third','middle third','top third'],right=False)
b['direction_band']=pd.cut(b.directionality,[-np.inf,-.5,0,.5,np.inf],labels=['≤-0.5','-0.5–<0','0–<0.5','≥0.5'],right=False)
b.to_csv(OUT/'short_grid_with_opening_features.csv',index=False)
print('Matched feature rows:',b.or_width.notna().sum(),'| source fills:',len(b),'| primary with RVOL:',(b.primary&b.feature_ready).sum())
p=b[b.primary&b.feature_ready].copy()
if p.empty:raise RuntimeError('No primary fills with complete prior mornings')
p['week_monday']=pd.to_datetime(p.date)-pd.to_timedelta(pd.to_datetime(p.date).dt.weekday,unit='D')
def report(d,keys):
    if d.empty:return pd.DataFrame()
    q=d.assign(w=d.result.eq('win').astype(int),l=d.result.eq('loss').astype(int),z=d.result.eq('neutral').astype(int))
    x=q.groupby(keys,observed=True,dropna=False).agg(fills=('R','size'),wins=('w','sum'),losses=('l','sum'),neutrals=('z','sum'),
         mean_R=('R','mean'),net_R=('R','sum'),tickers=('ticker','nunique'),dates=('date','nunique'),weeks=('week_monday','nunique')).reset_index()
    td=q.groupby(keys,observed=True,dropna=False).apply(lambda v:v[['ticker','date']].drop_duplicates().shape[0],include_groups=False).rename('ticker_days').reset_index()
    x=x.merge(td,on=keys,validate='one_to_one');x['win_rate_pct']=100*x.wins/x.fills
    assert (x.wins+x.losses+x.neutrals==x.fills).all()
    return x.round({'mean_R':3,'net_R':2,'win_rate_pct':1})
plan=['entry_method','stop_pct','target_pct']
for name,keys in [('baseline',plan),('rvol',plan+['rvol_band']),('dollar_rvol',plan+['dollar_rvol_band']),
                  ('or_close',plan+['close_band']),('or_directionality',plan+['direction_band']),
                  ('rvol_x_close',plan+['rvol_band','close_band'])]:
    t=report(p,keys);t.to_csv(OUT/(name+'.csv'),index=False)
    if name in ('baseline','rvol','or_close','or_directionality'):
        print(name,'1% stop/2% target');display(t[t.stop_pct.eq(.01)&t.target_pct.eq(.02)])


## Check date concentration and repeat fills

The main prior hypothesis is whether activity plus an OR close near its low helps short breakouts. Evaluate weekly distribution, first fill per ticker-day per independent plan, and removal of Nov 20 and Oct 12/14/16. Keep every band even when it disappoints. Only a later-week prospective replay can support a trading cutoff.

In [ ]:
for name,q in [('by_week',p),('first_fill',p.sort_values('entry_time').drop_duplicates(['ticker','date']+plan)),
               ('without_standout_dates',p[~p.date.isin(['2026-11-20','2026-10-12','2026-10-14','2026-10-16'])])]:
    for feature in ('rvol_band','close_band','direction_band'):
        keys=plan+(['week_monday'] if name=='by_week' else [])+[feature]
        report(q,keys).to_csv(OUT/(name+'_'+feature+'.csv'),index=False)
print('Results:',OUT)
print('Premarket bars are coverage only. Next studies: premarket high/low proximity at 10:00, first boundary touched, VWAP→OR and OR→VWAP paths with chronological signals and fills, conditional on data coverage.')